# North Carolina Primary Election Results - March 3, 2026 - Data cleaning, analysis, & mapping

We are going to create maps from the election results from the North Carolina Primary Election that occurred on March 3, 2026. 

**NC Board of Elections** handles tallying and certifying all election results and provides downloadable results for each election. 

**NC Board of Elections** also provides geographic shape files for all voting precincts in all 100 counties of North Carolina. Note: These precincts were recently redrawn and certified in December of 2025. Make sure you have the latest shape files that have the most up-to-date precincts.

## Analysis
This analysis and mapping project will strictly use the three counties that make up the **"Triangle"** i.e. **Wake, Durham, and Orange counties**.

### Cleaning data
Before we merge the election results with geospatial shape files, we need to confirm that the precinct id's on both data sets match. We will be merging data on precinct id and county name. If there are any mismatches, investigate why, and clean it up.

For example, in the Triangle counties analysis, Durham county has precinct ids as 01, 02, 03, ..., 09. On the geospatial shape files, these precinct ids showed as above, 01, 02, 03, ..., 09. However, in the election results data, these same precinct ids showed as 1, 2, 3, ..., 9, which cause a mismatch when attempting to merge the two data sets.

Next step is to merge the two data sets together.

Data pipeline steps
- Read election results csv
- filter for contest
- summarize contest (winner, runner-up, margin, etc)
- join with precinct geodata
- save as geojson
- import with mapbox gl js

In [1]:
import sys

print(sys.executable)

/Users/binguyen/.venvs/lede/bin/python


In [387]:
import os
import geopandas as gpd
import pandas as pd
import pydash

In [1025]:
script_dir = os.path.dirname(os.path.abspath('analysis.ipynb'))

In [389]:
csv_file_path = os.path.join(script_dir, '..', 'input', 'nc_primary_election_results_pct_20260303_RAW.csv')

In [390]:
results = pd.read_csv(csv_file_path)

## Filter election results by contest of choice

In [391]:
sorted(results["Contest Name"].unique())

['ALAMANCE COUNTY BOARD OF COMMISSIONERS (DEM)',
 'ALAMANCE COUNTY BOARD OF COMMISSIONERS (REP)',
 'ALAMANCE COUNTY CLERK OF SUPERIOR COURT (REP)',
 'ALAMANCE COUNTY SHERIFF (REP)',
 'ALEXANDER COUNTY BOARD OF COMMISSIONERS (REP)',
 'ALEXANDER COUNTY BOARD OF EDUCATION DISTRICT 02 (REP)',
 'ALEXANDER COUNTY CLERK OF SUPERIOR COURT (REP)',
 'ALEXANDER COUNTY REGISTER OF DEEDS (REP)',
 'ALLEGHANY COUNTY BOARD OF COMMISSIONERS (REP)',
 'ANSON COUNTY BOARD OF COMMISSIONERS DISTRICT 02 (REP)',
 'ANSON COUNTY BOARD OF EDUCATION AT-LARGE (DEM)',
 'ANSON COUNTY SHERIFF (DEM)',
 'ASHE COUNTY BOARD OF COMMISSIONERS (REP)',
 'ASHE COUNTY BOARD OF EDUCATION (REP)',
 'ASHE COUNTY SHERIFF (REP)',
 'AVERY COUNTY BOARD OF COMMISSIONERS (REP)',
 'AVERY COUNTY BOARD OF EDUCATION',
 'AVERY COUNTY CLERK OF SUPERIOR COURT (REP)',
 'BEAUFORT COUNTY BOARD OF COMMISSIONERS (REP)',
 'BEAUFORT COUNTY BOARD OF EDUCATION DISTRICT 02 (REP)',
 'BEAUFORT COUNTY BOARD OF EDUCATION DISTRICT 04 (REP)',
 'BEAUFORT COUNT

In [1125]:
contest_name = 'US SENATE (REP)'

In [1126]:
contest_df = results[results["Contest Name"] == contest_name]
contest_df

,County,Election Date,Precinct,Contest Group ID,Contest Type,Contest Name,Choice,Choice Party,Vote For,Election Day,Early Voting,Absentee by Mail,Provisional,Total Votes,Real Precinct
4,BUNCOMBE,03/03/2026,20.1,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,0,1,0,0,1,Y
13,BUNCOMBE,03/03/2026,23.2,2149,S,US SENATE (REP),Donald M. (Don) Brown,REP,1,1,1,0,0,2,Y
20,BUNCOMBE,03/03/2026,24.1,2149,S,US SENATE (REP),Richard Dansie,REP,1,4,4,0,0,8,Y
25,BUNCOMBE,03/03/2026,26.1,2149,S,US SENATE (REP),Thomas Johnson,REP,1,2,0,0,0,2,Y
27,BUNCOMBE,03/03/2026,28.1,2149,S,US SENATE (REP),Thomas Johnson,REP,1,1,1,0,0,2,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103499,WARREN,03/03/2026,11,2149,S,US SENATE (REP),Margot Dupre,REP,1,0,0,0,0,0,Y
103507,WASHINGTON,03/03/2026,LM,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,1,0,0,0,1,Y
103511,WASHINGTON,03/03/2026,P2,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,2,0,0,0,2,Y
103515,WASHINGTON,03/03/2026,SC,2149,S,US SENATE (REP),Michael Whatley,REP,1,57,0,0,0,57,Y


In [1127]:
# List of unique counties in chosen contest
contest_counties = contest_df["County"].unique()

In [1128]:
contest_counties

<StringArray>
[    'BUNCOMBE',     'CALDWELL',       'CAMDEN',     'CARTERET',
       'BLADEN',    'BRUNSWICK',     'BEAUFORT',       'BERTIE',
        'AVERY',        'BURKE',     'CABARRUS',      'FORSYTH',
    'GRANVILLE',       'GREENE',     'GUILFORD',     'FRANKLIN',
       'GASTON',   'MONTGOMERY',        'MOORE',         'PITT',
         'POLK',       'ONSLOW',       'ORANGE',   'PASQUOTANK',
       'PENDER',       'PERSON',     'JOHNSTON',        'JONES',
          'LEE',       'LENOIR',      'LINCOLN',  'MECKLENBURG',
     'MITCHELL',         'NASH',      'MADISON',       'MARTIN',
     'MCDOWELL',         'WAKE',       'WILSON',       'YADKIN',
       'YANCEY',        'UNION',        'VANCE',      'CASWELL',
      'CATAWBA',      'HAYWOOD',    'HENDERSON',      'HALIFAX',
      'HARNETT',       'GRAHAM',     'RANDOLPH',     'RICHMOND',
      'ROBESON',  'NEW HANOVER',        'WAYNE',       'WILKES',
      'WATAUGA',   'WASHINGTON',       'WARREN',      'TYRRELL',
      'CHAT

In [1129]:
# Now, filter the election results. First by "Real Precinct" == Y. 
# We will only map Real Precincts. See notes for data definitions.
contest_results_geo_df = contest_df[contest_df["Real Precinct"] == "Y"].copy()

In [1130]:
contest_results_geo_df.shape

(18431, 15)

In [1131]:
# Number of unique pairs of precincts and counties
# Different counties can share the same precinct ids e.g. Durham county, precinct 21 & Chatham county, precinct 21
# So we check for unique pairs
unique_pair_counts = contest_results_geo_df[['County', 'Precinct']].value_counts()

In [1132]:
unique_pair_counts

County       Precinct
BUNCOMBE     20.1        7
             23.2        7
             24.1        7
             26.1        7
             28.1        7
                        ..
ASHE         16          7
GUILFORD     H24         7
NORTHAMPTON  LAKE G      7
FORSYTH      14          7
ASHE         9           7
Name: count, Length: 2633, dtype: int64

In [1133]:
# Note: counts for each precinct should match the number of candidates in the contest
# Contest results data is one row per candidate in each precinct. We will group by precinct further below
# Validate manually using google spreadsheet
unique_pair_counts.groupby("County").size()

County
ALAMANCE     39
ALEXANDER    10
ALLEGHANY     4
ANSON         9
ASHE         17
             ..
WAYNE        28
WILKES       25
WILSON       24
YADKIN       12
YANCEY       11
Name: count, Length: 100, dtype: int64

### Get total votes for each candidate at the contest-level

In [1134]:
candidate_summary = (
    contest_results_geo_df
    .groupby("Choice", as_index=False)["Total Votes"]
    .sum()
    .rename(columns={
        "Choice": "candidate",
        "Total Votes": "votes"
    })
)

In [1135]:
candidate_summary["share"] = (
    candidate_summary["votes"]
    / candidate_summary["votes"].sum()
    * 100
)

In [1136]:
candidate_summary

,candidate,votes,share
0,Donald M. (Don) Brown,82382,15.626683
1,Elizabeth A. Temple,20047,3.802628
2,Margot Dupre,12417,2.355327
3,Michael Whatley,340816,64.647905
4,Michele Morrow,29371,5.571257
5,Richard Dansie,12654,2.400282
6,Thomas Johnson,29501,5.595916


In [1137]:
output_file_path = os.path.join(script_dir, '..', 'data', 'processed', f'{pydash.snake_case(contest_name)}_contest_candidate_summary.csv')

In [1138]:
candidate_summary.to_csv(output_file_path, index=False)

## Summarize contest results for MapBox

We will calculate and label results for our maps.
- Create a unique id from county and precincts
- Total votes in contest per precinct
- Winner
- Runner-up
- Margin of victory points
- Margin of victory percentage
- Each candidates vote share percentage
- Et. al.

In [1139]:
# Counts for each candidate represents the number of precincts that participated in the contest
# Validate with pair_counts
contest_results_geo_df["Choice"].value_counts()

Choice
Elizabeth A. Temple      2633
Donald M. (Don) Brown    2633
Richard Dansie           2633
Thomas Johnson           2633
Michael Whatley          2633
Michele Morrow           2633
Margot Dupre             2633
Name: count, dtype: int64

In [1140]:
# Check if Total Votes has any missing values i.e. precincts that did not participate
contest_results_geo_df["Total Votes"].isna().sum()

np.int64(0)

In [1142]:
contest_results_geo_df[contest_results_geo_df["Total Votes"] == 0]

,County,Election Date,Precinct,Contest Group ID,Contest Type,Contest Name,Choice,Choice Party,Vote For,Election Day,Early Voting,Absentee by Mail,Provisional,Total Votes,Real Precinct
250,BERTIE,03/03/2026,SN,2149,S,US SENATE (REP),Michele Morrow,REP,1,0,0,0,0,0,Y
262,BLADEN,03/03/2026,P10,2149,S,US SENATE (REP),Richard Dansie,REP,1,0,0,0,0,0,Y
297,BUNCOMBE,03/03/2026,34.1,2149,S,US SENATE (REP),Margot Dupre,REP,1,0,0,0,0,0,Y
607,FORSYTH,03/03/2026,205,2149,S,US SENATE (REP),Richard Dansie,REP,1,0,0,0,0,0,Y
650,GREENE,03/03/2026,BEAR,2149,S,US SENATE (REP),Thomas Johnson,REP,1,0,0,0,0,0,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103033,TYRRELL,03/03/2026,14,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,0,0,0,0,0,Y
103037,TYRRELL,03/03/2026,16,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,0,0,0,0,0,Y
103155,WAKE,03/03/2026,01-19,2149,S,US SENATE (REP),Margot Dupre,REP,1,0,0,0,0,0,Y
103185,WAKE,03/03/2026,01-41,2149,S,US SENATE (REP),Richard Dansie,REP,1,0,0,0,0,0,Y


In [1141]:
# Check if Total Votes is numeric
contest_results_geo_df["Total Votes"].dtype

dtype('int64')

### Create a unique id from precinct id and county name
Precinct ids are not truly unique. Different counties can have the same id e.g. Durham and Chatham both have precinct 15

In [1143]:
contest_results_geo_df["County"].dtype

<StringDtype(storage='python', na_value=nan)>

In [1144]:
contest_results_geo_df["Precinct"].dtype

<StringDtype(storage='python', na_value=nan)>

In [1145]:
contest_results_geo_df["county_precinct"] = contest_results_geo_df["County"] + '_' + contest_results_geo_df["Precinct"]

In [1146]:
contest_results_geo_df

,County,Election Date,Precinct,Contest Group ID,Contest Type,Contest Name,Choice,Choice Party,Vote For,Election Day,Early Voting,Absentee by Mail,Provisional,Total Votes,Real Precinct,county_precinct
4,BUNCOMBE,03/03/2026,20.1,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,0,1,0,0,1,Y,BUNCOMBE_20.1
13,BUNCOMBE,03/03/2026,23.2,2149,S,US SENATE (REP),Donald M. (Don) Brown,REP,1,1,1,0,0,2,Y,BUNCOMBE_23.2
20,BUNCOMBE,03/03/2026,24.1,2149,S,US SENATE (REP),Richard Dansie,REP,1,4,4,0,0,8,Y,BUNCOMBE_24.1
25,BUNCOMBE,03/03/2026,26.1,2149,S,US SENATE (REP),Thomas Johnson,REP,1,2,0,0,0,2,Y,BUNCOMBE_26.1
27,BUNCOMBE,03/03/2026,28.1,2149,S,US SENATE (REP),Thomas Johnson,REP,1,1,1,0,0,2,Y,BUNCOMBE_28.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103499,WARREN,03/03/2026,11,2149,S,US SENATE (REP),Margot Dupre,REP,1,0,0,0,0,0,Y,WARREN_11
103507,WASHINGTON,03/03/2026,LM,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,1,0,0,0,1,Y,WASHINGTON_LM
103511,WASHINGTON,03/03/2026,P2,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,2,0,0,0,2,Y,WASHINGTON_P2
103515,WASHINGTON,03/03/2026,SC,2149,S,US SENATE (REP),Michael Whatley,REP,1,57,0,0,0,57,Y,WASHINGTON_SC


### Get total votes casted in each precinct

In [1147]:
# Calculate total votes cast in each precinct
precinct_totals = (
    contest_results_geo_df
    .groupby("county_precinct")["Total Votes"]
    .sum()
    .rename("contest_votes")
)

In [1148]:
precinct_totals

county_precinct
ALAMANCE_03C     416
ALAMANCE_03N     354
ALAMANCE_03N2    129
ALAMANCE_03SE    339
ALAMANCE_03SM    310
                ... 
YANCEY_07 BRU     60
YANCEY_08 CRA    299
YANCEY_09 SOU    191
YANCEY_10 PEN     89
YANCEY_11 PRI    223
Name: contest_votes, Length: 2633, dtype: int64

In [1149]:
contest_results_geo_df = contest_results_geo_df.merge(
    precinct_totals,
    on="county_precinct",
    how="left"
)

In [1150]:
contest_results_geo_df

,County,Election Date,Precinct,Contest Group ID,Contest Type,Contest Name,Choice,Choice Party,Vote For,Election Day,Early Voting,Absentee by Mail,Provisional,Total Votes,Real Precinct,county_precinct,contest_votes
0,BUNCOMBE,03/03/2026,20.1,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,0,1,0,0,1,Y,BUNCOMBE_20.1,27
1,BUNCOMBE,03/03/2026,23.2,2149,S,US SENATE (REP),Donald M. (Don) Brown,REP,1,1,1,0,0,2,Y,BUNCOMBE_23.2,23
2,BUNCOMBE,03/03/2026,24.1,2149,S,US SENATE (REP),Richard Dansie,REP,1,4,4,0,0,8,Y,BUNCOMBE_24.1,124
3,BUNCOMBE,03/03/2026,26.1,2149,S,US SENATE (REP),Thomas Johnson,REP,1,2,0,0,0,2,Y,BUNCOMBE_26.1,38
4,BUNCOMBE,03/03/2026,28.1,2149,S,US SENATE (REP),Thomas Johnson,REP,1,1,1,0,0,2,Y,BUNCOMBE_28.1,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18426,WARREN,03/03/2026,11,2149,S,US SENATE (REP),Margot Dupre,REP,1,0,0,0,0,0,Y,WARREN_11,18
18427,WASHINGTON,03/03/2026,LM,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,1,0,0,0,1,Y,WASHINGTON_LM,66
18428,WASHINGTON,03/03/2026,P2,2149,S,US SENATE (REP),Elizabeth A. Temple,REP,1,2,0,0,0,2,Y,WASHINGTON_P2,27
18429,WASHINGTON,03/03/2026,SC,2149,S,US SENATE (REP),Michael Whatley,REP,1,57,0,0,0,57,Y,WASHINGTON_SC,86


### Summarize the contest results
Identify the winner and runner-up in each precinct. Then, create a new dataframe with one precinct per row for MapBox.


In [1151]:
contest_results_geo_df.rename(columns={"Total Votes": "candidate_votes"}, inplace=True)

In [1152]:
ranked = contest_results_geo_df.sort_values(["county_precinct", "candidate_votes"], ascending=[True, False]).copy()

In [1153]:
# Add rankings
ranked["rank"] = (
    ranked.groupby("county_precinct").cumcount() + 1
)

In [1154]:
first_second_df = ranked[ranked["rank"].isin([1,2])].copy()

In [1155]:
first_second_df

,County,Election Date,Precinct,Contest Group ID,Contest Type,Contest Name,Choice,Choice Party,Vote For,Election Day,Early Voting,Absentee by Mail,Provisional,candidate_votes,Real Precinct,county_precinct,contest_votes,rank
2224,ALAMANCE,03/03/2026,03C,2149,S,US SENATE (REP),Michael Whatley,REP,1,188,96,5,0,289,Y,ALAMANCE_03C,416,1
6897,ALAMANCE,03/03/2026,03C,2149,S,US SENATE (REP),Donald M. (Don) Brown,REP,1,31,7,0,0,38,Y,ALAMANCE_03C,416,2
2210,ALAMANCE,03/03/2026,03N,2149,S,US SENATE (REP),Michael Whatley,REP,1,147,96,5,0,248,Y,ALAMANCE_03N,354,1
2174,ALAMANCE,03/03/2026,03N,2149,S,US SENATE (REP),Donald M. (Don) Brown,REP,1,25,9,0,0,34,Y,ALAMANCE_03N,354,2
8084,ALAMANCE,03/03/2026,03N2,2149,S,US SENATE (REP),Michael Whatley,REP,1,76,13,0,0,89,Y,ALAMANCE_03N2,129,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672,YANCEY,03/03/2026,09 SOU,2149,S,US SENATE (REP),Donald M. (Don) Brown,REP,1,13,9,0,0,22,Y,YANCEY_09 SOU,191,2
17690,YANCEY,03/03/2026,10 PEN,2149,S,US SENATE (REP),Michael Whatley,REP,1,45,22,0,0,67,Y,YANCEY_10 PEN,89,1
18043,YANCEY,03/03/2026,10 PEN,2149,S,US SENATE (REP),Michele Morrow,REP,1,4,4,0,0,8,Y,YANCEY_10 PEN,89,2
14904,YANCEY,03/03/2026,11 PRI,2149,S,US SENATE (REP),Michael Whatley,REP,1,63,86,0,0,149,Y,YANCEY_11 PRI,223,1


In [1156]:
winner_df = (
    first_second_df[first_second_df["rank"] == 1]
        [["Contest Name", "County", "Precinct", "county_precinct", "contest_votes", "Choice", "candidate_votes"]].rename(columns={
        "Choice": "winner",
        "candidate_votes": "winner_votes"
    })
)

In [1157]:
winner_df

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes
2224,US SENATE (REP),ALAMANCE,03C,ALAMANCE_03C,416,Michael Whatley,289
2210,US SENATE (REP),ALAMANCE,03N,ALAMANCE_03N,354,Michael Whatley,248
8084,US SENATE (REP),ALAMANCE,03N2,ALAMANCE_03N2,129,Michael Whatley,89
2193,US SENATE (REP),ALAMANCE,03SE,ALAMANCE_03SE,339,Michael Whatley,252
6861,US SENATE (REP),ALAMANCE,03SM,ALAMANCE_03SM,310,Michael Whatley,193
...,...,...,...,...,...,...,...
9898,US SENATE (REP),YANCEY,07 BRU,YANCEY_07 BRU,60,Michael Whatley,38
12988,US SENATE (REP),YANCEY,08 CRA,YANCEY_08 CRA,299,Michael Whatley,194
12990,US SENATE (REP),YANCEY,09 SOU,YANCEY_09 SOU,191,Michael Whatley,119
17690,US SENATE (REP),YANCEY,10 PEN,YANCEY_10 PEN,89,Michael Whatley,67


In [1158]:
runner_up_df = (
    first_second_df[first_second_df["rank"] == 2]
        [["county_precinct", "Choice", "candidate_votes"]].rename(columns={
        "Choice": "runner_up",
        "candidate_votes": "runner_up_votes"
    })
)

In [1159]:
runner_up_df

,county_precinct,runner_up,runner_up_votes
6897,ALAMANCE_03C,Donald M. (Don) Brown,38
2174,ALAMANCE_03N,Donald M. (Don) Brown,34
2211,ALAMANCE_03N2,Donald M. (Don) Brown,15
6822,ALAMANCE_03SE,Donald M. (Don) Brown,27
6868,ALAMANCE_03SM,Donald M. (Don) Brown,51
...,...,...,...
14869,YANCEY_07 BRU,Michele Morrow,6
14902,YANCEY_08 CRA,Donald M. (Don) Brown,35
672,YANCEY_09 SOU,Donald M. (Don) Brown,22
18043,YANCEY_10 PEN,Michele Morrow,8


In [1160]:
contest_summary_df = winner_df.merge(
    runner_up_df,
    on="county_precinct",
    how="left"
)

In [1161]:
contest_summary_df

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes,runner_up,runner_up_votes
0,US SENATE (REP),ALAMANCE,03C,ALAMANCE_03C,416,Michael Whatley,289,Donald M. (Don) Brown,38
1,US SENATE (REP),ALAMANCE,03N,ALAMANCE_03N,354,Michael Whatley,248,Donald M. (Don) Brown,34
2,US SENATE (REP),ALAMANCE,03N2,ALAMANCE_03N2,129,Michael Whatley,89,Donald M. (Don) Brown,15
3,US SENATE (REP),ALAMANCE,03SE,ALAMANCE_03SE,339,Michael Whatley,252,Donald M. (Don) Brown,27
4,US SENATE (REP),ALAMANCE,03SM,ALAMANCE_03SM,310,Michael Whatley,193,Donald M. (Don) Brown,51
...,...,...,...,...,...,...,...,...,...
2628,US SENATE (REP),YANCEY,07 BRU,YANCEY_07 BRU,60,Michael Whatley,38,Michele Morrow,6
2629,US SENATE (REP),YANCEY,08 CRA,YANCEY_08 CRA,299,Michael Whatley,194,Donald M. (Don) Brown,35
2630,US SENATE (REP),YANCEY,09 SOU,YANCEY_09 SOU,191,Michael Whatley,119,Donald M. (Don) Brown,22
2631,US SENATE (REP),YANCEY,10 PEN,YANCEY_10 PEN,89,Michael Whatley,67,Michele Morrow,8


### Calculate vote shares for winner and runner-up
Vote Share: the percentage of total votes cast that a specific political party or candidate receives in an election.

In [1162]:
contest_summary_df["winner_vote_share"] = (
    (contest_summary_df["winner_votes"] /
    contest_summary_df["contest_votes"]) * 100
)

In [1163]:
contest_summary_df["runner_up_vote_share"] = (
    (contest_summary_df["runner_up_votes"] /
    contest_summary_df["contest_votes"]) * 100
)

In [1164]:
contest_summary_df

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share
0,US SENATE (REP),ALAMANCE,03C,ALAMANCE_03C,416,Michael Whatley,289,Donald M. (Don) Brown,38,69.471154,9.134615
1,US SENATE (REP),ALAMANCE,03N,ALAMANCE_03N,354,Michael Whatley,248,Donald M. (Don) Brown,34,70.056497,9.604520
2,US SENATE (REP),ALAMANCE,03N2,ALAMANCE_03N2,129,Michael Whatley,89,Donald M. (Don) Brown,15,68.992248,11.627907
3,US SENATE (REP),ALAMANCE,03SE,ALAMANCE_03SE,339,Michael Whatley,252,Donald M. (Don) Brown,27,74.336283,7.964602
4,US SENATE (REP),ALAMANCE,03SM,ALAMANCE_03SM,310,Michael Whatley,193,Donald M. (Don) Brown,51,62.258065,16.451613
...,...,...,...,...,...,...,...,...,...,...,...
2628,US SENATE (REP),YANCEY,07 BRU,YANCEY_07 BRU,60,Michael Whatley,38,Michele Morrow,6,63.333333,10.000000
2629,US SENATE (REP),YANCEY,08 CRA,YANCEY_08 CRA,299,Michael Whatley,194,Donald M. (Don) Brown,35,64.882943,11.705686
2630,US SENATE (REP),YANCEY,09 SOU,YANCEY_09 SOU,191,Michael Whatley,119,Donald M. (Don) Brown,22,62.303665,11.518325
2631,US SENATE (REP),YANCEY,10 PEN,YANCEY_10 PEN,89,Michael Whatley,67,Michele Morrow,8,75.280899,8.988764


### Calculate margin of victory
MOV: Difference between the vote share of the winning candidate and the second-place candidate. 

For example, if Candidate A wins with 55% of the vote and Candidate B receives 45%, the margin of victory is 10 percentage points. 

In [1165]:
contest_summary_df["margin_of_victory"] = (
    contest_summary_df["winner_vote_share"] - contest_summary_df["runner_up_vote_share"]
)

In [1166]:
contest_summary_df

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share,margin_of_victory
0,US SENATE (REP),ALAMANCE,03C,ALAMANCE_03C,416,Michael Whatley,289,Donald M. (Don) Brown,38,69.471154,9.134615,60.336538
1,US SENATE (REP),ALAMANCE,03N,ALAMANCE_03N,354,Michael Whatley,248,Donald M. (Don) Brown,34,70.056497,9.604520,60.451977
2,US SENATE (REP),ALAMANCE,03N2,ALAMANCE_03N2,129,Michael Whatley,89,Donald M. (Don) Brown,15,68.992248,11.627907,57.364341
3,US SENATE (REP),ALAMANCE,03SE,ALAMANCE_03SE,339,Michael Whatley,252,Donald M. (Don) Brown,27,74.336283,7.964602,66.371681
4,US SENATE (REP),ALAMANCE,03SM,ALAMANCE_03SM,310,Michael Whatley,193,Donald M. (Don) Brown,51,62.258065,16.451613,45.806452
...,...,...,...,...,...,...,...,...,...,...,...,...
2628,US SENATE (REP),YANCEY,07 BRU,YANCEY_07 BRU,60,Michael Whatley,38,Michele Morrow,6,63.333333,10.000000,53.333333
2629,US SENATE (REP),YANCEY,08 CRA,YANCEY_08 CRA,299,Michael Whatley,194,Donald M. (Don) Brown,35,64.882943,11.705686,53.177258
2630,US SENATE (REP),YANCEY,09 SOU,YANCEY_09 SOU,191,Michael Whatley,119,Donald M. (Don) Brown,22,62.303665,11.518325,50.785340
2631,US SENATE (REP),YANCEY,10 PEN,YANCEY_10 PEN,89,Michael Whatley,67,Michele Morrow,8,75.280899,8.988764,66.292135


### Get results for all candidates for each precinct


In [1169]:
def calc_vote_share(candidate_votes, contest_votes):
    if contest_votes > 0:
        return float((candidate_votes / contest_votes) * 100)
    if contest_votes == 0:
        return 0

In [1170]:
all_candidate_results = (
    contest_results_geo_df
    .sort_values(["county_precinct", "candidate_votes"], ascending=[True, False])
    .groupby("county_precinct")
    .apply(
        lambda g: [
            {
                "candidate": row["Choice"],
                "votes": int(row["candidate_votes"]),
                "share": calc_vote_share(row["candidate_votes"], row["contest_votes"])
            }
            for _, row in g.iterrows()
        ],
        include_groups=False
    )
)

In [1171]:
all_candidate_results

county_precinct
ALAMANCE_03C     [{'candidate': 'Michael Whatley', 'votes': 289...
ALAMANCE_03N     [{'candidate': 'Michael Whatley', 'votes': 248...
ALAMANCE_03N2    [{'candidate': 'Michael Whatley', 'votes': 89,...
ALAMANCE_03SE    [{'candidate': 'Michael Whatley', 'votes': 252...
ALAMANCE_03SM    [{'candidate': 'Michael Whatley', 'votes': 193...
                                       ...                        
YANCEY_07 BRU    [{'candidate': 'Michael Whatley', 'votes': 38,...
YANCEY_08 CRA    [{'candidate': 'Michael Whatley', 'votes': 194...
YANCEY_09 SOU    [{'candidate': 'Michael Whatley', 'votes': 119...
YANCEY_10 PEN    [{'candidate': 'Michael Whatley', 'votes': 67,...
YANCEY_11 PRI    [{'candidate': 'Michael Whatley', 'votes': 149...
Length: 2633, dtype: object

In [1172]:
contest_summary_df[contest_summary_df["county_precinct"] == "WAKE_12-08"]

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share,margin_of_victory
2406,US SENATE (REP),WAKE,12-08,WAKE_12-08,155,Michael Whatley,104,Donald M. (Don) Brown,25,67.096774,16.129032,50.967742


In [1174]:
contest_summary_df["all_candidate_results"] = contest_summary_df["county_precinct"].map(all_candidate_results)

In [1175]:
contest_summary_df

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share,margin_of_victory,all_candidate_results
0,US SENATE (REP),ALAMANCE,03C,ALAMANCE_03C,416,Michael Whatley,289,Donald M. (Don) Brown,38,69.471154,9.134615,60.336538,"[{'candidate': 'Michael Whatley', 'votes': 289..."
1,US SENATE (REP),ALAMANCE,03N,ALAMANCE_03N,354,Michael Whatley,248,Donald M. (Don) Brown,34,70.056497,9.604520,60.451977,"[{'candidate': 'Michael Whatley', 'votes': 248..."
2,US SENATE (REP),ALAMANCE,03N2,ALAMANCE_03N2,129,Michael Whatley,89,Donald M. (Don) Brown,15,68.992248,11.627907,57.364341,"[{'candidate': 'Michael Whatley', 'votes': 89,..."
3,US SENATE (REP),ALAMANCE,03SE,ALAMANCE_03SE,339,Michael Whatley,252,Donald M. (Don) Brown,27,74.336283,7.964602,66.371681,"[{'candidate': 'Michael Whatley', 'votes': 252..."
4,US SENATE (REP),ALAMANCE,03SM,ALAMANCE_03SM,310,Michael Whatley,193,Donald M. (Don) Brown,51,62.258065,16.451613,45.806452,"[{'candidate': 'Michael Whatley', 'votes': 193..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2628,US SENATE (REP),YANCEY,07 BRU,YANCEY_07 BRU,60,Michael Whatley,38,Michele Morrow,6,63.333333,10.000000,53.333333,"[{'candidate': 'Michael Whatley', 'votes': 38,..."
2629,US SENATE (REP),YANCEY,08 CRA,YANCEY_08 CRA,299,Michael Whatley,194,Donald M. (Don) Brown,35,64.882943,11.705686,53.177258,"[{'candidate': 'Michael Whatley', 'votes': 194..."
2630,US SENATE (REP),YANCEY,09 SOU,YANCEY_09 SOU,191,Michael Whatley,119,Donald M. (Don) Brown,22,62.303665,11.518325,50.785340,"[{'candidate': 'Michael Whatley', 'votes': 119..."
2631,US SENATE (REP),YANCEY,10 PEN,YANCEY_10 PEN,89,Michael Whatley,67,Michele Morrow,8,75.280899,8.988764,66.292135,"[{'candidate': 'Michael Whatley', 'votes': 67,..."


In [1180]:
contest_summary_df[contest_summary_df['contest_votes'] == 0]

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share,margin_of_victory,all_candidate_results
705,US SENATE (REP),DURHAM,12,DURHAM_12,0,Michele Morrow,0,Michael Whatley,0,NaN,NaN,NaN,"[{'candidate': 'Michele Morrow', 'votes': 0, '..."
740,US SENATE (REP),DURHAM,41,DURHAM_41,0,Michael Whatley,0,Margot Dupre,0,NaN,NaN,NaN,"[{'candidate': 'Michael Whatley', 'votes': 0, ..."
754,US SENATE (REP),DURHAM,55-49,DURHAM_55-49,0,Michael Whatley,0,Richard Dansie,0,NaN,NaN,NaN,"[{'candidate': 'Michael Whatley', 'votes': 0, ..."
768,US SENATE (REP),EDGECOMBE,1202,EDGECOMBE_1202,0,Michael Whatley,0,Donald M. (Don) Brown,0,NaN,NaN,NaN,"[{'candidate': 'Michael Whatley', 'votes': 0, ..."
806,US SENATE (REP),FORSYTH,301,FORSYTH_301,0,Donald M. (Don) Brown,0,Elizabeth A. Temple,0,NaN,NaN,NaN,"[{'candidate': 'Donald M. (Don) Brown', 'votes..."
808,US SENATE (REP),FORSYTH,303,FORSYTH_303,0,Elizabeth A. Temple,0,Michele Morrow,0,NaN,NaN,NaN,"[{'candidate': 'Elizabeth A. Temple', 'votes':..."
1066,US SENATE (REP),GUILFORD,G68,GUILFORD_G68,0,Thomas Johnson,0,Michael Whatley,0,NaN,NaN,NaN,"[{'candidate': 'Thomas Johnson', 'votes': 0, '..."
1622,US SENATE (REP),MECKLENBURG,56,MECKLENBURG_56,0,Michele Morrow,0,Richard Dansie,0,NaN,NaN,NaN,"[{'candidate': 'Michele Morrow', 'votes': 0, '..."
2097,US SENATE (REP),SAMPSON,CLCE,SAMPSON_CLCE,0,Richard Dansie,0,Elizabeth A. Temple,0,NaN,NaN,NaN,"[{'candidate': 'Richard Dansie', 'votes': 0, '..."
2098,US SENATE (REP),SAMPSON,CLEA,SAMPSON_CLEA,0,Thomas Johnson,0,Richard Dansie,0,NaN,NaN,NaN,"[{'candidate': 'Thomas Johnson', 'votes': 0, '..."


In [1181]:
# Needed for crosswalk right now. Will eventually reorganize files

In [1177]:
output_file_path = os.path.join(script_dir, '..', 'data', 'processed', f'{pydash.snake_case(contest_name)}_contest_summary.csv')

In [1178]:
contest_summary_df.to_csv(output_file_path, index=False)

### Calculate Opacity if needed

### Bring in precinct shape file and merge with our contest's summary data
Results should produce a geodataframe with geometry for each precinct

In [1092]:
shp_file_path = os.path.join(script_dir, '..', 'data', 'raw', 'geography', 'SBE_PRECINCTS_20251212', 'SBE_PRECINCTS_20251212.shp')

In [1093]:
precincts_gdf = gpd.read_file(shp_file_path)

In [1094]:
# Filter precinct GeoDataFrame by counties in the chosen contest
contest_precincts_gdf = precincts[precincts["county_nam"].isin(contest_counties)]

In [1095]:
contest_precincts_gdf

,id,county_id,prec_id,enr_desc,county_nam,Shape_Leng,Shape_Area,of_prec_id,geometry
59,105,92,01-05,01-05,WAKE,14500.105940,1.325901e+07,NaN,"POLYGON ((2104962.991 746518.459, 2104960.007 ..."
71,110,92,01-09,01-09,WAKE,12038.234692,8.161909e+06,NaN,"POLYGON ((2106835.489 746138.993, 2106712.756 ..."
114,99,92,01-01,01-01,WAKE,16835.331171,1.549578e+07,NaN,"POLYGON ((2100440.458 741338.285, 2100440.556 ..."
116,101,92,01-02,01-02,WAKE,24437.015415,2.445547e+07,NaN,"POLYGON ((2096969.047 743377.041, 2096967.151 ..."
117,102,92,01-03,01-03,WAKE,21790.186717,2.242057e+07,NaN,"POLYGON ((2096391.936 746636.928, 2096320.174 ..."
...,...,...,...,...,...,...,...,...,...
2623,3141,92,13-12,13-12,WAKE,40066.315909,4.649720e+07,NaN,"POLYGON ((2121521.829 766440.325, 2121587.911 ..."
2624,3142,92,12-12,12-12,WAKE,46001.066465,8.927364e+07,NaN,"POLYGON ((2074291.413 679114.557, 2074060.621 ..."
2625,3143,92,12-13,12-13,WAKE,50225.930446,1.275286e+08,NaN,"POLYGON ((2067423.477 669347.8, 2067409.868 66..."
2626,3144,92,05-09,05-09,WAKE,26574.404656,4.180295e+07,NaN,"POLYGON ((2047460.895 752794.873, 2047335.208 ..."


In [1096]:
# Number of rows = number of all precincts in contest counties
# Note: not all precincts in the entire county participate in every contest
contest_precincts_gdf.shape

(231, 9)

In [1097]:
contest_precincts_gdf["county_nam"].unique()

<StringArray>
['WAKE', 'GRANVILLE']
Length: 2, dtype: str

In [98]:
# Note: not all precincts in each county participate in certain contests
# e.g. US House of Represenetatives District 04 (Dem) includes some but not all of Wake and Chatham

In [102]:
# Note: Some precincts need format cleaning.
# e.g. Durham has precincts 01-09. On the precinct geo file, these precinct ids are 01-09. 
# On the election results data, these precinct ids are 1-9
# Precinct ids must match in order to join both data sets. Otherwise mismatches will occur and votes count will be inaccuracy

In [1098]:
# Function to normalize the 1 and 01 precinct id issue in Durham

def normalize_precinct_id(x):
    x = str(x).strip()

    if x.isdigit():
        return x.zfill(2)

    return x

In [1099]:
# We will make a new column on both dfs that creates a normalized two digit precinct id
# i.e. both will have precinct ids "01", "02", etc.
# The function above also account for precincts that are alpha characters i.e. Orange County
contest_precincts_gdf["precinct_join"] = contest_precincts_gdf["prec_id"]

contest_summary_df["precinct_join"] = contest_summary_df["Precinct"].apply(
    normalize_precinct_id
)

In [1100]:
contest_summary_df

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share,margin_of_victory,all_candidate_results,precinct_join
0,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,ANTI,GRANVILLE_ANTI,101,Chris Stock,63,Cheryl Caulfield,38,62.376238,37.623762,24.752475,"[{'candidate': 'Chris Stock', 'votes': 63, 'sh...",ANTI
1,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,BERE,GRANVILLE_BERE,240,Chris Stock,167,Cheryl Caulfield,73,69.583333,30.416667,39.166667,"[{'candidate': 'Chris Stock', 'votes': 167, 's...",BERE
2,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,BTNR,GRANVILLE_BTNR,218,Chris Stock,161,Cheryl Caulfield,57,73.853211,26.146789,47.706422,"[{'candidate': 'Chris Stock', 'votes': 161, 's...",BTNR
3,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,CORI,GRANVILLE_CORI,306,Chris Stock,202,Cheryl Caulfield,104,66.013072,33.986928,32.026144,"[{'candidate': 'Chris Stock', 'votes': 202, 's...",CORI
4,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,CRDL,GRANVILLE_CRDL,107,Chris Stock,75,Cheryl Caulfield,32,70.093458,29.906542,40.186916,"[{'candidate': 'Chris Stock', 'votes': 75, 'sh...",CRDL
5,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,CRDM,GRANVILLE_CRDM,472,Chris Stock,332,Cheryl Caulfield,140,70.338983,29.661017,40.677966,"[{'candidate': 'Chris Stock', 'votes': 332, 's...",CRDM
6,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,EAOX,GRANVILLE_EAOX,135,Chris Stock,104,Cheryl Caulfield,31,77.037037,22.962963,54.074074,"[{'candidate': 'Chris Stock', 'votes': 104, 's...",EAOX
7,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,MTEN,GRANVILLE_MTEN,579,Chris Stock,391,Cheryl Caulfield,188,67.530225,32.469775,35.060449,"[{'candidate': 'Chris Stock', 'votes': 391, 's...",MTEN
8,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,OKHL,GRANVILLE_OKHL,163,Chris Stock,111,Cheryl Caulfield,52,68.098160,31.901840,36.196319,"[{'candidate': 'Chris Stock', 'votes': 111, 's...",OKHL
9,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,SALM,GRANVILLE_SALM,234,Chris Stock,166,Cheryl Caulfield,68,70.940171,29.059829,41.880342,"[{'candidate': 'Chris Stock', 'votes': 166, 's...",SALM


In [1101]:
# We will use both precinct id and county name to join to have more precision
# Durham's "01" could potentially match with Wake's "01-01". Joining with both
# normalized county name and precinct id will prevent this.
contest_precincts_gdf["county_join"] = (
    contest_precincts_gdf["county_nam"]
    .str.strip()
    .str.upper()
)

contest_summary_df["county_join"] = (
    contest_summary_df["County"]
    .str.strip()
    .str.upper()
)

In [1102]:
contest_precincts_gdf

,id,county_id,prec_id,enr_desc,county_nam,Shape_Leng,Shape_Area,of_prec_id,geometry,precinct_join,county_join
59,105,92,01-05,01-05,WAKE,14500.105940,1.325901e+07,NaN,"POLYGON ((2104962.991 746518.459, 2104960.007 ...",01-05,WAKE
71,110,92,01-09,01-09,WAKE,12038.234692,8.161909e+06,NaN,"POLYGON ((2106835.489 746138.993, 2106712.756 ...",01-09,WAKE
114,99,92,01-01,01-01,WAKE,16835.331171,1.549578e+07,NaN,"POLYGON ((2100440.458 741338.285, 2100440.556 ...",01-01,WAKE
116,101,92,01-02,01-02,WAKE,24437.015415,2.445547e+07,NaN,"POLYGON ((2096969.047 743377.041, 2096967.151 ...",01-02,WAKE
117,102,92,01-03,01-03,WAKE,21790.186717,2.242057e+07,NaN,"POLYGON ((2096391.936 746636.928, 2096320.174 ...",01-03,WAKE
...,...,...,...,...,...,...,...,...,...,...,...
2623,3141,92,13-12,13-12,WAKE,40066.315909,4.649720e+07,NaN,"POLYGON ((2121521.829 766440.325, 2121587.911 ...",13-12,WAKE
2624,3142,92,12-12,12-12,WAKE,46001.066465,8.927364e+07,NaN,"POLYGON ((2074291.413 679114.557, 2074060.621 ...",12-12,WAKE
2625,3143,92,12-13,12-13,WAKE,50225.930446,1.275286e+08,NaN,"POLYGON ((2067423.477 669347.8, 2067409.868 66...",12-13,WAKE
2626,3144,92,05-09,05-09,WAKE,26574.404656,4.180295e+07,NaN,"POLYGON ((2047460.895 752794.873, 2047335.208 ...",05-09,WAKE


In [1103]:
contest_summary_df

,Contest Name,County,Precinct,county_precinct,contest_votes,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share,margin_of_victory,all_candidate_results,precinct_join,county_join
0,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,ANTI,GRANVILLE_ANTI,101,Chris Stock,63,Cheryl Caulfield,38,62.376238,37.623762,24.752475,"[{'candidate': 'Chris Stock', 'votes': 63, 'sh...",ANTI,GRANVILLE
1,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,BERE,GRANVILLE_BERE,240,Chris Stock,167,Cheryl Caulfield,73,69.583333,30.416667,39.166667,"[{'candidate': 'Chris Stock', 'votes': 167, 's...",BERE,GRANVILLE
2,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,BTNR,GRANVILLE_BTNR,218,Chris Stock,161,Cheryl Caulfield,57,73.853211,26.146789,47.706422,"[{'candidate': 'Chris Stock', 'votes': 161, 's...",BTNR,GRANVILLE
3,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,CORI,GRANVILLE_CORI,306,Chris Stock,202,Cheryl Caulfield,104,66.013072,33.986928,32.026144,"[{'candidate': 'Chris Stock', 'votes': 202, 's...",CORI,GRANVILLE
4,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,CRDL,GRANVILLE_CRDL,107,Chris Stock,75,Cheryl Caulfield,32,70.093458,29.906542,40.186916,"[{'candidate': 'Chris Stock', 'votes': 75, 'sh...",CRDL,GRANVILLE
5,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,CRDM,GRANVILLE_CRDM,472,Chris Stock,332,Cheryl Caulfield,140,70.338983,29.661017,40.677966,"[{'candidate': 'Chris Stock', 'votes': 332, 's...",CRDM,GRANVILLE
6,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,EAOX,GRANVILLE_EAOX,135,Chris Stock,104,Cheryl Caulfield,31,77.037037,22.962963,54.074074,"[{'candidate': 'Chris Stock', 'votes': 104, 's...",EAOX,GRANVILLE
7,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,MTEN,GRANVILLE_MTEN,579,Chris Stock,391,Cheryl Caulfield,188,67.530225,32.469775,35.060449,"[{'candidate': 'Chris Stock', 'votes': 391, 's...",MTEN,GRANVILLE
8,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,OKHL,GRANVILLE_OKHL,163,Chris Stock,111,Cheryl Caulfield,52,68.098160,31.901840,36.196319,"[{'candidate': 'Chris Stock', 'votes': 111, 's...",OKHL,GRANVILLE
9,NC STATE SENATE DISTRICT 18 (REP),GRANVILLE,SALM,GRANVILLE_SALM,234,Chris Stock,166,Cheryl Caulfield,68,70.940171,29.059829,41.880342,"[{'candidate': 'Chris Stock', 'votes': 166, 's...",SALM,GRANVILLE


In [1104]:
contest_summary_gdf = contest_precincts_gdf.merge(
    contest_summary_df,
    left_on=["county_join", "precinct_join"],
    right_on=["county_join", "precinct_join"],
    how="left",
    indicator=True
)

In [1105]:
type(contest_summary_gdf)

geopandas.geodataframe.GeoDataFrame

In [1106]:
contest_summary_gdf

,id,county_id,prec_id,enr_desc,county_nam,Shape_Leng,Shape_Area,of_prec_id,geometry,precinct_join,...,contest_votes,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share,margin_of_victory,all_candidate_results,_merge
0,105,92,01-05,01-05,WAKE,14500.105940,1.325901e+07,NaN,"POLYGON ((2104962.991 746518.459, 2104960.007 ...",01-05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,110,92,01-09,01-09,WAKE,12038.234692,8.161909e+06,NaN,"POLYGON ((2106835.489 746138.993, 2106712.756 ...",01-09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,99,92,01-01,01-01,WAKE,16835.331171,1.549578e+07,NaN,"POLYGON ((2100440.458 741338.285, 2100440.556 ...",01-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,101,92,01-02,01-02,WAKE,24437.015415,2.445547e+07,NaN,"POLYGON ((2096969.047 743377.041, 2096967.151 ...",01-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,102,92,01-03,01-03,WAKE,21790.186717,2.242057e+07,NaN,"POLYGON ((2096391.936 746636.928, 2096320.174 ...",01-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226,3141,92,13-12,13-12,WAKE,40066.315909,4.649720e+07,NaN,"POLYGON ((2121521.829 766440.325, 2121587.911 ...",13-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
227,3142,92,12-12,12-12,WAKE,46001.066465,8.927364e+07,NaN,"POLYGON ((2074291.413 679114.557, 2074060.621 ...",12-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
228,3143,92,12-13,12-13,WAKE,50225.930446,1.275286e+08,NaN,"POLYGON ((2067423.477 669347.8, 2067409.868 66...",12-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
229,3144,92,05-09,05-09,WAKE,26574.404656,4.180295e+07,NaN,"POLYGON ((2047460.895 752794.873, 2047335.208 ...",05-09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [1107]:
contest_summary_gdf.shape

(231, 25)

In [1108]:
# both count should match number of precincts that participated in the contest (after filtering Real Precincts == Y)
# total of both + left_only should match total number of precincts in all counties that participated in the contest
contest_summary_gdf["_merge"].value_counts()

_merge
left_only     186
both           45
right_only      0
Name: count, dtype: int64

In [1109]:
list(contest_summary_gdf)

['id',
 'county_id',
 'prec_id',
 'enr_desc',
 'county_nam',
 'Shape_Leng',
 'Shape_Area',
 'of_prec_id',
 'geometry',
 'precinct_join',
 'county_join',
 'Contest Name',
 'County',
 'Precinct',
 'county_precinct',
 'contest_votes',
 'winner',
 'winner_votes',
 'runner_up',
 'runner_up_votes',
 'winner_vote_share',
 'runner_up_vote_share',
 'margin_of_victory',
 'all_candidate_results',
 '_merge']

### Create a map key for MapBox

In [1111]:
contest_summary_gdf["county_precinct"] = (
    contest_summary_gdf["county_join"]
    + "_"
    + contest_summary_gdf["precinct_join"]
)

In [1113]:
contest_summary_gdf["county_precinct"]

0      WAKE_01-05
1      WAKE_01-09
2      WAKE_01-01
3      WAKE_01-02
4      WAKE_01-03
          ...    
226    WAKE_13-12
227    WAKE_12-12
228    WAKE_12-13
229    WAKE_05-09
230    WAKE_05-10
Name: county_precinct, Length: 231, dtype: str

In [1114]:
contest_summary_gdf.rename(columns={"county_precinct": "map_key"}, inplace=True)

In [1115]:
contest_summary_gdf["map_key"]

0      WAKE_01-05
1      WAKE_01-09
2      WAKE_01-01
3      WAKE_01-02
4      WAKE_01-03
          ...    
226    WAKE_13-12
227    WAKE_12-12
228    WAKE_12-13
229    WAKE_05-09
230    WAKE_05-10
Name: map_key, Length: 231, dtype: str

### For filtering out non-participating precincts on the map

In [1116]:
contest_summary_gdf["participated"] = (
    contest_summary_gdf["contest_votes"]
    .fillna(0)
    .gt(0)
)

In [1117]:
# True  = precinct participated in the contest
# False = precinct did not
contest_summary_gdf["participated"]

0      False
1      False
2      False
3      False
4      False
       ...  
226    False
227    False
228    False
229    False
230    False
Name: participated, Length: 231, dtype: bool

In [1118]:
contest_summary_gdf.crs

<Projected CRS: EPSG:2264>
Name: NAD83 / North Carolina (ftUS)
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - North Carolina - counties of Alamance; Alexander; Alleghany; Anson; Ashe; Avery; Beaufort; Bertie; Bladen; Brunswick; Buncombe; Burke; Cabarrus; Caldwell; Camden; Carteret; Caswell; Catawba; Chatham; Cherokee; Chowan; Clay; Cleveland; Columbus; Craven; Cumberland; Currituck; Dare; Davidson; Davie; Duplin; Durham; Edgecombe; Forsyth; Franklin; Gaston; Gates; Graham; Granville; Greene; Guilford; Halifax; Harnett; Haywood; Henderson; Hertford; Hoke; Hyde; Iredell; Jackson; Johnston; Jones; Lee; Lenoir; Lincoln; Macon; Madison; Martin; McDowell; Mecklenburg; Mitchell; Montgomery; Moore; Nash; New Hanover; Northampton; Onslow; Orange; Pamlico; Pasquotank; Pender; Perquimans; Person; Pitt; Polk; Randolph; Richmond; Robeson; Rockingham; Rowan; Rutherford; Sampson; Scotland; Stanly; Stokes; Sur

In [1119]:
contest_summary_web = contest_summary_gdf.to_crs("EPSG:4326")

### Copy/ Paste contest bounds in contest.js

In [1120]:
participating = contest_summary_web[
    contest_summary_web["participated"]
].copy()

west, south, east, north = participating.total_bounds

contest_bounds = [
    [west, south],
    [east, north]
]

print(contest_bounds)

[[np.float64(-78.80762299092058), np.float64(35.783509999663366)], [np.float64(-78.25371099138387), np.float64(36.54254100036947)]]


In [1121]:
contest_summary_web

,id,county_id,prec_id,enr_desc,county_nam,Shape_Leng,Shape_Area,of_prec_id,geometry,precinct_join,...,winner,winner_votes,runner_up,runner_up_votes,winner_vote_share,runner_up_vote_share,margin_of_victory,all_candidate_results,_merge,participated
0,105,92,01-05,01-05,WAKE,14500.105940,1.325901e+07,NaN,"POLYGON ((-78.64603 35.80064, -78.64604 35.800...",01-05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1,110,92,01-09,01-09,WAKE,12038.234692,8.161909e+06,NaN,"POLYGON ((-78.63972 35.79958, -78.64013 35.799...",01-09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
2,99,92,01-01,01-01,WAKE,16835.331171,1.549578e+07,NaN,"POLYGON ((-78.66134 35.78646, -78.66134 35.786...",01-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
3,101,92,01-02,01-02,WAKE,24437.015415,2.445547e+07,NaN,"POLYGON ((-78.67302 35.79209, -78.67303 35.791...",01-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
4,102,92,01-03,01-03,WAKE,21790.186717,2.242057e+07,NaN,"POLYGON ((-78.67493 35.80105, -78.67517 35.801...",01-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226,3141,92,13-12,13-12,WAKE,40066.315909,4.649720e+07,NaN,"POLYGON ((-78.58991 35.8552, -78.58969 35.8552...",13-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
227,3142,92,12-12,12-12,WAKE,46001.066465,8.927364e+07,NaN,"POLYGON ((-78.75004 35.61572, -78.75081 35.615...",12-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
228,3143,92,12-13,12-13,WAKE,50225.930446,1.275286e+08,NaN,"POLYGON ((-78.77322 35.58893, -78.77326 35.589...",12-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
229,3144,92,05-09,05-09,WAKE,26574.404656,4.180295e+07,NaN,"POLYGON ((-78.83991 35.8183, -78.84034 35.8182...",05-09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False


In [1122]:
processed_file_path = os.path.join(script_dir, '..', 'data', 'processed', f'{pydash.snake_case(contest_name)}_contest_summary.geojson')
map_data_file_path = os.path.join(script_dir, '..', 'map', 'data', f'{pydash.snake_case(contest_name)}_contest_summary.geojson')

In [1124]:
contest_summary_web.to_file(processed_file_path, driver="GeoJSON")
contest_summary_web.to_file(map_data_file_path, driver="GeoJSON")